# TP 4 — Décorateurs Python

## Étape 1 — Chronométrer une fonction

In [1]:
import time
from functools import wraps

def chronometrer(f):
    @wraps(f)
    def wrapper(*args, **kwargs):
        debut = time.perf_counter()
        resultat = f(*args, **kwargs)
        print(f"{f.__name__} : {time.perf_counter()-debut:.4f} s")
        return resultat
    return wrapper

@chronometrer
def petit_calcul():
    time.sleep(0.2)
    return sum(range(1000))
print(petit_calcul())

petit_calcul : 0.2003 s
499500


## Étape 2 — Mettre le résultat en majuscules

In [2]:
def majuscules(f):
    @wraps(f)
    def wrapper(*args, **kwargs):
        resultat = f(*args, **kwargs)
        return resultat.upper() if isinstance(resultat, str) else resultat
    return wrapper

@majuscules
def saluer(nom):
    return "Bonjour " + nom
print(saluer("Miora"))

BONJOUR MIORA


## Étape 3 — Compter les appels

In [3]:
def compter_appels(f):
    @wraps(f)
    def wrapper(*args, **kwargs):
        wrapper.appels += 1
        return f(*args, **kwargs)
    wrapper.appels = 0
    return wrapper

@compter_appels
def carre(n):
    return n*n
print(carre(4), carre(5), carre(6))
print("Appels :", carre.appels)

16 25 36
Appels : 3


## Étape 4 — Vérifier les arguments positifs

In [4]:
def verifier_positifs(f):
    @wraps(f)
    def wrapper(a, b):
        if a < 0 or b < 0:
            raise ValueError("Arguments positifs ou nuls requis")
        return f(a, b)
    return wrapper

@verifier_positifs
def addition(a, b):
    return a+b
print(addition(4, 8))
try:
    addition(-1, 8)
except ValueError as e:
    print("Erreur :", e)

12
Erreur : Arguments positifs ou nuls requis


## Étape 5 — Décorateur avec unité

In [5]:
def chronometrer_unite(unite="s"):
    facteurs = {"s": 1, "ms": 1000}
    if unite not in facteurs:
        raise ValueError("Choisir s ou ms")
    def decorateur(f):
        @wraps(f)
        def wrapper(*args, **kwargs):
            debut = time.perf_counter()
            resultat = f(*args, **kwargs)
            print(f"{f.__name__} : {(time.perf_counter()-debut)*facteurs[unite]:.2f} {unite}")
            return resultat
        return wrapper
    return decorateur

@chronometrer_unite("ms")
def somme(n):
    return sum(range(n))
print(somme(10000))

somme : 0.77 ms
49995000


## Étape 6 — Fibonacci avec mémorisation

In [6]:
from functools import lru_cache

def memoriser(f):
    cache = {}
    @wraps(f)
    def wrapper(n):
        if n not in cache:
            cache[n] = f(n)
        return cache[n]
    wrapper.cache = cache
    return wrapper

@memoriser
def fib(n):
    if n < 2: return n
    return fib(n-1) + fib(n-2)

@lru_cache(maxsize=None)
def fib_lru(n):
    if n < 2: return n
    return fib_lru(n-1) + fib_lru(n-2)

print(fib(30), "éléments mémorisés :", len(fib.cache))
print(fib_lru(30), fib_lru.cache_info())

832040 éléments mémorisés : 31
832040 CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)


## Étape 7 — MiniFlask

In [7]:
class MiniFlask:
    def __init__(self):
        self.routes = {}
    def route(self, chemin):
        def decorateur(f):
            if chemin in self.routes:
                raise ValueError("Route déjà enregistrée")
            self.routes[chemin] = f
            return f
        return decorateur
    def afficher_routes(self):
        for chemin in sorted(self.routes):
            print(chemin)

app = MiniFlask()
@app.route("/")
def accueil():
    return "Bienvenue"
@app.route("/aide")
def aide():
    return "Aide"
app.afficher_routes()
print(app.routes["/"]())

/
/aide
Bienvenue
